## Example 1: Basic Value and Error Creation

In [ ]:
import outcome

# Example 1: Basic outcome types
print("Example 1 - Basic Value and Error creation")

# Create Value outcomes
success = outcome.Value(42)
print(f"Success outcome: {success}")
print(f"Extract value: {success.unwrap()}")

# Create Error outcomes
failure = outcome.Error(ValueError("Something failed"))
print(f"\nError outcome: {failure}")
print(f"Error type: {type(failure.error).__name__}")

# Type checking
print(f"\nIs success a Value? {isinstance(success, outcome.Value)}")
print(f"Is failure an Error? {isinstance(failure, outcome.Error)}")
# unwrap() may only be called once per outcome, so check the stored value directly
assert success.value == 42
assert isinstance(failure.error, ValueError)

print("Example 1 completed")

## Example 2: Capturing Function Results

In [ ]:
import outcome

# Example 2: Using capture to wrap function calls
print("Example 2 - Capturing function results")

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("Cannot divide by zero")
    return a / b

# Capture successful call
result1 = outcome.capture(divide, 10, 2)
print(f"10 / 2 = {result1.unwrap()}")

# Capture failed call
result2 = outcome.capture(divide, 10, 0)
print(f"10 / 0 error: {type(result2.error).__name__}")

# Batch processing
operations = [(10, 2), (15, 3), (8, 0), (20, 4)]
results = [outcome.capture(divide, a, b) for a, b in operations]

for i, (a, b) in enumerate(operations):
    if isinstance(results[i], outcome.Value):
        print(f"{a}/{b} = {results[i].unwrap()}")
    else:
        print(f"{a}/{b} failed")

print("Example 2 completed")

## Example 3: Error Handling Without Exceptions

In [ ]:
import outcome

# Example 3: Error handling patterns
print("Example 3 - Error handling without exceptions")

def safe_int_parse(value):
    """Parse integer with outcome instead of try/except."""
    return outcome.capture(int, value)

# Test with various inputs
inputs = ["123", "456", "not_a_number", "789", "12.34"]

for inp in inputs:
    result = safe_int_parse(inp)
    if isinstance(result, outcome.Value):
        print(f"'{inp}' -> {result.unwrap()}")
    else:
        print(f"'{inp}' -> Error: {type(result.error).__name__}")

# Count successes
all_results = [safe_int_parse(inp) for inp in inputs]
successes = sum(1 for r in all_results if isinstance(r, outcome.Value))
print(f"\nSuccessful parses: {successes}/{len(inputs)}")
assert successes == 3  # "not_a_number" and "12.34" are not valid int() input

print("Example 3 completed")

## Example 4: Chaining Operations with Outcomes

In [ ]:
import outcome

# Example 4: Chaining operations
print("Example 4 - Chaining operations")

def validate_positive(n):
    if n <= 0:
        raise ValueError("Number must be positive")
    return n

def calculate_sqrt(n):
    return n ** 0.5

def process_number(value):
    """Chain: parse -> validate -> calculate."""
    # Step 1: Parse
    parsed = outcome.capture(int, value)
    if isinstance(parsed, outcome.Error):
        return parsed
    
    # Step 2: Validate
    validated = outcome.capture(validate_positive, parsed.unwrap())
    if isinstance(validated, outcome.Error):
        return validated
    
    # Step 3: Calculate
    return outcome.capture(calculate_sqrt, validated.unwrap())

# Test chaining
test_values = ["16", "25", "-4", "invalid", "9"]

for val in test_values:
    result = process_number(val)
    if isinstance(result, outcome.Value):
        print(f"sqrt({val}) = {result.unwrap():.2f}")
    else:
        print(f"{val}: {type(result.error).__name__}")

print("Example 4 completed")

## Example 5: Outcome for Result Aggregation

In [ ]:
import outcome

# Example 5: Aggregating results
print("Example 5 - Result aggregation")

def safe_operation(x, y, op):
    """Safely perform arithmetic operation."""
    operations = {
        'add': lambda: x + y,
        'sub': lambda: x - y,
        'mul': lambda: x * y,
        'div': lambda: x / y
    }
    return outcome.capture(operations[op])

# Batch operations
ops = [
    (10, 5, 'add'), (10, 5, 'sub'),
    (10, 5, 'mul'), (10, 5, 'div'),
    (10, 0, 'div')  # Will fail
]

results = [safe_operation(x, y, op) for x, y, op in ops]

# Aggregate statistics
total_ops = len(results)
successful = [r for r in results if isinstance(r, outcome.Value)]
failed = [r for r in results if isinstance(r, outcome.Error)]

print(f"Operations: {total_ops}")
print(f"Successful: {len(successful)}")
print(f"Failed: {len(failed)}")

if successful:
    values = [r.unwrap() for r in successful]
    print(f"Results: {values}")

print("Example 5 completed")